In [2]:
import pandas as pd
from fpdf import FPDF
import os
import glob
from pathlib import Path

CRIAR DATA FRAME:

INSERIR CAMINHO DA PLANILHA ABAIXO COM A SEGUINTE ESTRUTURA: CAMINHO DO ARQUIVO, NOME DA PLANILHA

PARA GERAR MAIS DE UM MATERIAL BASTA CRIAR MAIS DE UM DATA_FRAME

In [ ]:
df_quadradinho = pd.read_excel(r"C:\Users\Victor.figueiredo\Downloads\PRODUTOS.xlsx","nome_aba",dtype = {'PRODUTO' : str,'COR_PRODUTO': str})

df_quadradinho['DATA_ENTRADA'] = pd.to_datetime(df_quadradinho['DATA_ENTRADA']).dt.strftime('%d/%m/%Y')

df_quadradinho = df_quadradinho.sort_values(by ='LINHA', ascending = True)

MAPEAMENTO FOTOS: ALTERAR O CAMINHO PARA MAPEAR AS FOTOS DA COLEÇÃO DESEJADA

In [ ]:
# 1. Defina o caminho base (o ** e recursive=True buscam em todas as subpastas)

caminho_col1 = r'\**\*.jpg'

caminho_col2 = r'\**\*.jpg'



mapa_colecao1 = {os.path.basename(f): f for f in glob.glob(caminho_col1, recursive=True)}

mapa_colecao2 = {os.path.basename(f): f for f in glob.glob(caminho_col2, recursive=True)}



mapa_fotos = mapa_colecao1 | mapa_colecao2



print(f"Mapeamento concluído! {len(mapa_fotos)} fotos encontradas.")

In [ ]:
def gerar_catalogo(df,nome_aba):
    # criando página vazia
    pdf = FPDF(orientation='L', unit='mm', format='A4')
    pdf.add_page()

    # determinando parâmetros iniciais
    X_inicial = 10  # Posição horizontal inicial
    X_foto = 10  # Posição horizontal à ser incrementada
    Y_inicial = 15  # Posição vertical Inicial
    Y_foto = 15  # Posicação vertical à ser incrementada
    largura = 35  # Largura ocupada pela foto
    contador = 0  # Utilizado para pular linha e páginas no loop
    espacamento = 90  # Espaçamento vertical entre quadros
    espacamento_horizontal = 60 #Espaçamento horizontal entre quadros
    
    # loop de fotos
    for row in df.itertuples():
        pdf.set_fill_color(246, 234, 214)
        pdf.set_font('Helvetica', 'B', 6)
        pdf.set_xy(X_foto, Y_foto - 5)  # 5mm acima da posição da foto
        pdf.cell(w=largura, h=4, text=str(row.LINHA), new_x="RIGHT", new_y="TOP", align='C', fill='True')
        
        # FOTOS
        # 2. SE a foto existir, coloca ela
        nome_esperado = f"{row.PRODUTO}-{row.COR_PRODUTO}.jpg"

        nome_esperado_2 = f"{row.PRODUTO}.{row.COR_PRODUTO}.jpg"
        nome_final = nome_esperado if nome_esperado in mapa_fotos else nome_esperado_2

        caminho_foto_prod = mapa_fotos.get(nome_final)

    # 2. SE a foto existir, coloca ela
        if caminho_foto_prod is not None:
            try:
                pdf.image(caminho_foto_prod, x=X_foto, y=Y_foto, w=largura)  
            except Exception:
                print(f"Erro ao renderizar imagem de {row.PRODCOR}")
                # Se der erro na imagem, podemos desenhar um retângulo vazio no lugar
                pdf.rect(X_foto, Y_foto, largura, largura + 20)
                pdf.set_y(Y_foto + largura - 11)  # Seta o Y logo abaixo da altura da foto
                pdf.set_x(X_foto)
                pdf.set_font('Helvetica', '', 7)
                pdf.cell(w=largura, h=5, text="Imagem Não Encontrada", new_x="RIGHT", new_y="TOP", align='C')
        else:
            # Se der erro na imagem, podemos desenhar um retângulo vazio no lugar
            pdf.rect(X_foto, Y_foto, largura, largura + 20)
            pdf.set_y(Y_foto + largura - 11)  # Seta o Y logo abaixo da altura da foto
            pdf.set_x(X_foto)
            pdf.set_font('Helvetica', '', 7)
            pdf.cell(w=largura, h=5, text="Imagem Não Encontrada", new_x="RIGHT", new_y="TOP", align='C')

        ## PRODUTO
        # Ele usa a mesma coordenada X da foto para ficar alinhado
        pdf.set_y(Y_foto + largura + 20)  # Seta o Y logo abaixo da altura da foto
        pdf.set_x(X_foto)
        pdf.set_font('Helvetica', 'B', 7)
        pdf.cell(w=largura, h=5, text=row.PRODCOR, new_x="RIGHT", new_y="TOP", align='C')
        
        # DESC_PRODUTO
        pdf.set_y(Y_foto + largura + 25)
        pdf.set_x(X_foto)
        pdf.set_font('Helvetica', '', 6)
        pdf.cell(w=largura, h=3, text=row.DESC_PRODUTO, new_x="RIGHT", new_y="TOP", align='c')
        
        # DESC_COR_PRODUTO
        pdf.set_y(Y_foto + largura + 28)
        pdf.set_x(X_foto)
        pdf.set_font('Helvetica', '', 6)
        pdf.cell(w=largura, h=3, text=row.DESC_COR_PRODUTO, new_x="RIGHT", new_y="TOP", align='c')
        
        # DATA_CAPSULA
        pdf.set_y(Y_foto + largura + 31)
        pdf.set_x(X_foto)
        pdf.set_font('Helvetica', '', 6)
        pdf.cell(w=largura, h=3, text=f'Entrada: {row.DATA_ENTRADA}', new_x="RIGHT", new_y="TOP", align='c')
        
        # DIAS_VENDA
        pdf.set_y(Y_foto + largura + 34)
        pdf.set_x(X_foto)
        pdf.set_font('Helvetica', '', 6)
        pdf.cell(w=largura, h=3, text=f'Estoque: {str(row.ESTOQUE)}', new_x="RIGHT", new_y="TOP", align='c')

        contador = contador + 1
        
        if contador == 10:
            pdf.add_page()
            X_foto, Y_foto = X_inicial, Y_inicial
            contador = 0
        elif contador == 5:
            X_foto = X_inicial
            Y_foto = Y_foto + espacamento
        else:
            X_foto += espacamento_horizontal
    caminho = Path(r"C:\Imanges")
    nome_arquivo = f"{nome_aba}.pdf"
    caminho_arquivo = caminho / nome_arquivo
    pdf.output(str(caminho_arquivo))

AO GERAR O CATÁLOGO INSERIR: DATA_FRAMA, NOME DA PLANILHA

In [ ]:
gerar_catalogo(df_quadradinho,"nome_aba")
#gerar_catalogo(df_quadradinho2,"PLANILHA_2")
#gerar_catalogo(df_quadradinho3,"PLANILHA_3")